# Detección de Ciberataques con Redes Neuronales Profundas sobre CIC-IDS2017

**Proyecto ML – Avance** · Tema PR-36 · Maestría MMIA/MSDS – USFQ
**Autor:** Ronnie Almeida

| Ítem de la rúbrica | Sección |
|---|---|
| Introducción | 1 |
| EDA | 2 |
| Feature Extraction | 3 (no aplica: datos tabulares) |
| Feature Selection (filter) | 4 |
| Pipelines | 4 |
| Optimización de hiperparámetros + curva de aprendizaje | 5 |
| Comparación estadística de 2 técnicas ML | 6 |
| Visualización con t-SNE | 7 |
| Conclusiones | 8 |
| Referencias (IEEE) | 9 |

## 1. Introducción

Los ataques contra infraestructuras de red (denegación de servicio distribuida, escaneo de puertos, fuerza bruta, botnets e intrusiones web) siguen creciendo en volumen y sofisticación. Los sistemas de detección de intrusiones (IDS) tradicionales se basan en **firmas**: reconocen solo ataques ya conocidos y requieren mantenimiento manual constante. Para una organización, como un banco o un proveedor de servicios en la nube, un ataque no detectado implica indisponibilidad del servicio, fuga de información y pérdidas económicas y reputacionales.

Los IDS basados en **aprendizaje automático** aprenden a distinguir tráfico normal de tráfico malicioso a partir de estadísticas de los flujos de red (duración, número de paquetes, tamaños, flags TCP, tiempos entre llegadas, etc.), lo que permite generalizar a variantes de ataques. En particular, las **redes neuronales profundas** pueden modelar relaciones no lineales complejas entre estas características.

**Problema:** clasificar cada flujo de red como **benigno (0)** o **ataque (1)**.

**Dataset:** *CIC-IDS2017* del Canadian Institute for Cybersecurity [1]: ~2.8 millones de flujos etiquetados, generados durante 5 días en una red realista, con 78 características extraídas por CICFlowMeter y 14 tipos de ataque (DoS/DDoS, PortScan, FTP/SSH-Patator, Botnet, Web Attacks, Infiltration, Heartbleed).

**Objetivo de este avance:** construir un pipeline reproducible (limpieza → selección de características → modelo), optimizar hiperparámetros con validación cruzada repetida y comparar estadísticamente una **red neuronal profunda (MLP)** frente a un **Random Forest** como línea base.

**Importancia de la métrica:** en un IDS, un falso negativo (ataque no detectado) es costoso, pero demasiados falsos positivos generan *alert fatigue* en el equipo de operaciones. Por eso usamos **F1-score**, que balancea *precision* y *recall*, en lugar de *accuracy* (engañosa con clases desbalanceadas).

### 1.1 Configuración e importaciones

In [ ]:
# Nota (rúbrica - uso de IA): el esqueleto de este notebook se generó con apoyo de un
# asistente de IA (Claude). Prompt usado:
#   "Con el tema PR-36 (detección de ciberataques con redes neuronales profundas) ayúdame a
#    resolver el avance del proyecto: EDA, feature selection tipo filter en un pipeline de
#    sklearn, optimización de hiperparámetros con RepeatedStratifiedKFold, curva de aprendizaje,
#    comparación estadística (Wilcoxon) de 2 técnicas ML, t-SNE, conclusiones y referencias IEEE."
import os, glob, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, cross_validate, cross_val_score, learning_curve)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, f_classif
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, roc_auc_score)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
SEED = 42
np.random.seed(SEED)

# ---------------- Parámetros del experimento ----------------
N_BENIGN      = 20_000   # flujos benignos muestreados
MAX_PER_ATTACK = 2_000   # tope de flujos por tipo de ataque (conserva las clases raras)
CV_SPLITS, CV_REPEATS_TUNE = 5, 2     # tuning: 2 repeticiones x 5 folds
CMP_SPLITS, CMP_REPEATS    = 10, 10   # comparación: 10x10 (recomendado). Si es lento -> 5, 2
N_TSNE = 3_000
N_JOBS = -1

### 1.2 Carga de datos

En Google Colab el dataset se descarga desde Kaggle con `kagglehub` (versión *MachineLearningCVE* de CIC-IDS2017, 8 archivos CSV, uno por jornada). Alternativa: subir manualmente los CSV a la carpeta `data/`.

Como el dataset completo tiene ~2.8 M filas, se construye una **muestra estratificada**: 20 000 flujos benignos y hasta 2 000 por cada tipo de ataque, para conservar las clases minoritarias y mantener tiempos de entrenamiento razonables.

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "data")
if not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    try:
        import kagglehub  # en Colab: !pip install kagglehub (ya suele venir instalado)
        DATA_DIR = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
    except Exception as e:
        raise RuntimeError("No se pudo descargar desde Kaggle. Sube los CSV de CIC-IDS2017 a la carpeta 'data/'.") from e

files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print(f"{len(files)} archivos CSV encontrados:")
for f in files: print("  -", os.path.basename(f))

In [ ]:
def clean_label(s):
    # Algunos labels traen caracteres corruptos (p.ej. 'Web Attack � Brute Force')
    return re.sub(r"[^A-Za-z0-9 \-]+", "-", str(s)).replace(" - ", " ").strip()

parts, raw_counts = [], []
for f in files:
    df_f = pd.read_csv(f, low_memory=False, encoding_errors="replace")
    df_f.columns = df_f.columns.str.strip()
    df_f["Label"] = df_f["Label"].map(clean_label)
    df_f["Day"] = os.path.basename(f).split("-")[0]
    raw_counts.append(df_f["Label"].value_counts())
    parts.append(df_f)
full = pd.concat(parts, ignore_index=True); del parts
raw_counts = full["Label"].value_counts()
print("Tamaño del dataset completo:", full.shape)

rng = np.random.RandomState(SEED)
samples = []
for lab, g in full.groupby("Label"):
    n = N_BENIGN if lab == "BENIGN" else min(len(g), MAX_PER_ATTACK)
    samples.append(g.sample(n=min(n, len(g)), random_state=rng))
df = pd.concat(samples).sample(frac=1, random_state=SEED).reset_index(drop=True)
del full, samples
print("Tamaño de la muestra de trabajo:", df.shape)
df.head()

## 2. Análisis Exploratorio de Datos (EDA)

### 2.1 Distribución de clases (dataset completo vs. muestra)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
raw_counts.sort_values().plot.barh(ax=axes[0], color="#4C72B0", logx=True)
axes[0].set_title("Dataset completo (escala log)"); axes[0].set_xlabel("Número de flujos")
df["Label"].value_counts().sort_values().plot.barh(ax=axes[1], color="#DD8452", logx=True)
axes[1].set_title("Muestra de trabajo (escala log)"); axes[1].set_xlabel("Número de flujos")
plt.tight_layout(); plt.show()

print("Proporción benigno en el dataset completo: {:.1%}".format(raw_counts["BENIGN"] / raw_counts.sum()))
df["Attack"] = (df["Label"] != "BENIGN").astype(int)
print(df["Attack"].value_counts(normalize=True).rename({0: "Benigno", 1: "Ataque"}))

**Observación:** el dataset original tiene 2 830 743 flujos y está fuertemente desbalanceado: el 80,3 % es tráfico benigno. Además, algunas clases son extremadamente raras: Heartbleed (11 flujos), Web Attack Sql Injection (21) e Infiltration (36). El muestreo conserva los 15 tipos de tráfico y deja la tarea binaria prácticamente balanceada (≈ 50 % / 50 %). Aun así, se usa **F1** como métrica principal (accuracy ocultaría los errores en la clase ataque) y validación **estratificada por tipo de ataque**, para que las clases raras aparezcan en todos los conjuntos.

### 2.2 Tipos de datos, valores faltantes, infinitos y duplicados

In [ ]:
feat_cols = [c for c in df.columns if c not in ("Label", "Attack", "Day")]
X_num = df[feat_cols].apply(pd.to_numeric, errors="coerce")
print("Número de características:", len(feat_cols))
print("Tipos:", X_num.dtypes.value_counts().to_dict())
n_nan = X_num.isna().sum(); n_inf = np.isinf(X_num).sum()
print("\nColumnas con NaN:", n_nan[n_nan > 0].to_dict())
print("Columnas con ±inf:", n_inf[n_inf > 0].to_dict())
const_cols = [c for c in feat_cols if X_num[c].nunique(dropna=False) <= 1]
print(f"\nColumnas constantes ({len(const_cols)}):", const_cols)
dup_cols = [c for c in feat_cols if c.endswith(".1")]
print("Columnas duplicadas por nombre:", dup_cols)
print("Filas duplicadas:", df.duplicated(subset=feat_cols + ["Label"]).sum())

**Observación:** como reporta la literatura [2], CIC-IDS2017 contiene valores infinitos y NaN en `Flow Bytes/s` y `Flow Packets/s` (división por duración cero), columnas constantes (flags que nunca se activan) y una columna duplicada (`Fwd Header Length.1`). Las filas duplicadas se eliminan aquí; los infinitos/NaN y las columnas constantes se tratan **dentro del pipeline** (sección 4) para evitar *data leakage*.

In [ ]:
df = df.drop_duplicates(subset=feat_cols + ["Label"]).reset_index(drop=True)
feat_cols = [c for c in feat_cols if c not in dup_cols]
X = df[feat_cols].apply(pd.to_numeric, errors="coerce").astype("float64")
y = df["Attack"].values
labels_multi = df["Label"].values
print("Muestra sin duplicados:", X.shape, "| ataques:", y.sum())

**Observación:** se eliminaron 2 064 filas duplicadas y la muestra quedó en 38 129 flujos, de los cuales 18 337 son ataques (48,1 %). Casi el 90 % de los duplicados eran ataques: herramientas como DoS Hulk o PortScan generan flujos idénticos de forma masiva. Mantener esos duplicados inflaría artificialmente las métricas, porque el mismo flujo podría quedar a la vez en entrenamiento y en prueba.

### 2.3 Estadísticos descriptivos

In [ ]:
desc = X.replace([np.inf, -np.inf], np.nan).describe().T
desc["skew"] = X.replace([np.inf, -np.inf], np.nan).skew()
desc.sort_values("skew", ascending=False).head(15).round(2)

**Observación:** las variables tienen escalas muy distintas (flags entre 0 y 1 frente a `Flow Bytes/s`, con máximo ≈ 2·10⁹) y distribuciones con asimetría extrema: el coeficiente llega a 194,5 en `Total Backward Packets`, con la mediana muy por debajo de la media. Esto obliga a **estandarizar** antes de la red neuronal (sensible a la escala), mientras que el Random Forest es invariante a transformaciones monótonas.

### 2.4 Variables más informativas y su distribución por clase

In [ ]:
X_tmp = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())
X_tmp = X_tmp.loc[:, X_tmp.nunique() > 1]
mi = pd.Series(mutual_info_classif(X_tmp, y, random_state=SEED), index=X_tmp.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
mi.head(20).sort_values().plot.barh(ax=ax, color="#55A868")
ax.set_title("Top 20 características por información mutua con la etiqueta"); ax.set_xlabel("Información mutua")
plt.tight_layout(); plt.show()

top6 = mi.index[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), top6):
    sns.boxplot(x=np.where(y == 1, "Ataque", "Benigno"), y=np.log1p(X_tmp[c].clip(lower=0)), ax=ax,
                palette={"Benigno": "#4C72B0", "Ataque": "#C44E52"}, showfliers=False)
    ax.set_title(c); ax.set_ylabel("log(1 + x)")
plt.suptitle("Distribución (log) de las 6 variables más informativas por clase"); plt.tight_layout(); plt.show()

**Observación:** según la información mutua, la variable más informativa es `Destination Port`. El boxplot muestra que casi todos los ataques de la muestra apuntan a un único puerto (80, HTTP): la caja de la clase ataque colapsa en una línea. Esto no describe el *comportamiento* del ataque, sino cómo se diseñó el experimento de captura. Es un posible **atajo** (*shortcut learning*) que puede inflar el desempeño y no generalizar a otra red [14]. Su efecto se mide en la sección 5.5. Las variables de longitud de paquete (`Packet Length Std/Mean`, `Max Packet Length`) muestran que los ataques tienen medianas más bajas, pero mayor dispersión que el tráfico benigno.

### 2.5 Correlación entre variables

In [ ]:
top25 = mi.index[:25]
corr = X_tmp[top25].corr(method="spearman")
plt.figure(figsize=(13, 11))
sns.heatmap(corr, cmap="vlag", center=0, square=True, cbar_kws={"shrink": .7})
plt.title("Correlación de Spearman (top 25 variables por información mutua)"); plt.tight_layout(); plt.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).abs().stack()
print("Pares con |rho| > 0.95:", (upper > 0.95).sum())
upper.sort_values(ascending=False).head(10)

**Observación:** entre las 25 variables más informativas hay **31 pares con |ρ| > 0,95**, y cuatro de ellos son idénticos (ρ = 1): `Fwd Packet Length Mean` y `Avg Fwd Segment Size`, `Bwd Packet Length Mean` y `Avg Bwd Segment Size`, `Total Length of Fwd Packets` y `Subflow Fwd Bytes`, y `Total Length of Bwd Packets` y `Subflow Bwd Bytes`. CICFlowMeter calcula la misma magnitud con distintos nombres. Además, los tiempos (*IAT*, `Flow Duration`) se correlacionan negativamente con las tasas (`Flow Packets/s`), como es de esperar. Esto motiva la **selección de características**: reducir dimensionalidad sin perder información discriminante y acelerar el entrenamiento.

### 2.6 División entrenamiento / prueba

In [ ]:
X_train, X_test, y_train, y_test, lab_train, lab_test = train_test_split(
    X, y, labels_multi, test_size=0.2, stratify=labels_multi, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporción de ataques -> train: {:.3f}, test: {:.3f}".format(y_train.mean(), y_test.mean()))

El conjunto de prueba (20 %) se separa **antes** de cualquier ajuste y solo se usa al final. La estratificación se hace por tipo de ataque para que todas las clases aparezcan en ambos conjuntos.

## 3. Feature Extraction

**No aplica.** CIC-IDS2017 ya es un dataset **tabular**: las características de cada flujo fueron extraídas por CICFlowMeter a partir de las capturas *pcap*. No se trabaja con datos no estructurados (imágenes, texto, audio) ni con *transfer learning*.

## 4. Pipelines y Feature Selection (tipo *filter*)

Todo el preprocesamiento vive dentro de un `Pipeline` de scikit-learn [3], de modo que en cada fold de validación cruzada se ajusta **solo con los datos de entrenamiento** de ese fold (sin *data leakage*):

1. `FunctionTransformer`: reemplaza ±inf por NaN.
2. `SimpleImputer(median)`: imputa NaN.
3. `VarianceThreshold`: elimina columnas constantes (filtro).
4. `StandardScaler`: estandariza (necesario para la red neuronal).
5. `SelectKBest(f_classif)`: **feature selection tipo filter** basado en el estadístico F de ANOVA; `k` se trata como hiperparámetro.
6. Clasificador: `MLPClassifier` (red neuronal profunda) o `RandomForestClassifier`.

Se usa el estadístico F (en vez de información mutua) dentro del pipeline porque es determinista y mucho más rápido, lo que importa al repetirlo cientos de veces en la validación cruzada.

In [ ]:
def inf_to_nan(A):
    A = np.asarray(A, dtype="float64").copy()
    A[~np.isfinite(A)] = np.nan
    return A

def make_preprocessor():
    return [("inf2nan", FunctionTransformer(inf_to_nan)),
            ("imputer", SimpleImputer(strategy="median")),
            ("var",     VarianceThreshold(0.0)),
            ("scaler",  StandardScaler()),
            ("select",  SelectKBest(score_func=f_classif, k=30))]

pipe_mlp = Pipeline(make_preprocessor() + [
    ("clf", MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu", solver="adam",
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          max_iter=300, random_state=SEED))])

pipe_rf = Pipeline(make_preprocessor() + [
    ("clf", RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=SEED))])
pipe_mlp

In [ ]:
# Ranking de variables según el filtro (ajustado solo en train)
pre = Pipeline(make_preprocessor()[:-1]).fit(X_train, y_train)
kept = np.array(feat_cols)[pre.named_steps["var"].get_support()]
F, p = f_classif(pre.transform(X_train), y_train)
fscores = pd.Series(F, index=kept).sort_values(ascending=False)
plt.figure(figsize=(8, 7))
fscores.head(25).sort_values().plot.barh(color="#8172B3")
plt.title("Top 25 características según ANOVA F (filtro)"); plt.xlabel("Estadístico F")
plt.tight_layout(); plt.show()
print(f"Variables tras VarianceThreshold: {len(kept)} de {len(feat_cols)}")

**Observación:** `VarianceThreshold` elimina 8 de las 77 variables: las 6 de *Bulk* (`Fwd/Bwd Avg Bytes/Bulk`, etc.) y los flags `Bwd PSH Flags` y `Bwd URG Flags`, que nunca se activan. Quedan 69 variables candidatas. El ranking ANOVA-F (encabezado por `Bwd Packet Length Min` y `PSH Flag Count`) **no coincide** con el de información mutua de la sección 2.4. Es esperable, porque F solo mide diferencias de **medias** entre clases (relación lineal), mientras que la información mutua capta también relaciones **no lineales**, como la del puerto de destino. Por eso el número de variables `k` no se fija a mano, sino que se optimiza con validación cruzada.

## 5. Optimización de hiperparámetros

Se usa `GridSearchCV` con **`RepeatedStratifiedKFold` (5 folds × 2 repeticiones)** sobre el conjunto de entrenamiento [4]. No se usa *nested CV* por su costo computacional (como indica la rúbrica). Métrica de optimización: **F1** de la clase ataque.

### 5.1 Red neuronal profunda (MLP)

In [ ]:
cv_tune = RepeatedStratifiedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS_TUNE, random_state=SEED)

grid_mlp = {
    "select__k": [20, 40],
    "clf__hidden_layer_sizes": [(64, 32), (128, 64, 32)],
    "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],   # regularización L2
}
t0 = time.time()
gs_mlp = GridSearchCV(pipe_mlp, grid_mlp, scoring="f1", cv=cv_tune, n_jobs=N_JOBS, return_train_score=True)
gs_mlp.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.0f}s")
print("Mejores hiperparámetros MLP:", gs_mlp.best_params_)
print("F1 CV: {:.4f} ± {:.4f}".format(gs_mlp.best_score_, gs_mlp.cv_results_["std_test_score"][gs_mlp.best_index_]))

In [ ]:
res = pd.DataFrame(gs_mlp.cv_results_)
res["arch"] = res["param_clf__hidden_layer_sizes"].astype(str)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, k in zip(axes, grid_mlp["select__k"]):
    for (arch, g), col in zip(res[res["param_select__k"] == k].groupby("arch"), ["#4C72B0", "#55A868"]):
        g = g.sort_values("param_clf__alpha")
        a = g["param_clf__alpha"].astype(float)
        ax.plot(a, g["mean_test_score"], "o-", color=col, label=f"{arch} (validación)")
        ax.fill_between(a, g["mean_test_score"] - g["std_test_score"], g["mean_test_score"] + g["std_test_score"], color=col, alpha=.15)
        ax.plot(a, g["mean_train_score"], "--", color=col, alpha=.6, label=f"{arch} (entrenamiento)")
    ax.set_xscale("log"); ax.set_xlabel("alpha (regularización L2)"); ax.set_title(f"MLP, k = {k} características")
axes[0].set_ylabel("F1 (media ± std, 5x2 CV)"); axes[1].legend(fontsize=8)
plt.suptitle("Optimización de hiperparámetros del MLP"); plt.tight_layout(); plt.show()

**Observación:** la mejor configuración del MLP es la arquitectura profunda (128, 64, 32) con `alpha = 1e-3` y `k = 40`, con F1 = 0,9673 ± 0,0026.
- Con `k = 40` el F1 sube alrededor de 1,3 puntos respecto a `k = 20` en todas las configuraciones. La red se beneficia de tener más variables.
- La arquitectura de 3 capas supera a la de 2 en todos los casos.
- La regularización L2 casi no influye entre 10⁻⁵ y 10⁻², pero con `alpha = 0,1` el F1 cae: el modelo empieza a **subajustar**.
- La brecha entre entrenamiento y validación es menor a 0,005, así que no hay sobreajuste.

### 5.2 Random Forest (línea base)

In [ ]:
grid_rf = {
    "select__k": [10, 20, 40],
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [10, 20, None],
}
t0 = time.time()
gs_rf = GridSearchCV(pipe_rf, grid_rf, scoring="f1", cv=cv_tune, n_jobs=N_JOBS, return_train_score=True)
gs_rf.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.0f}s")
print("Mejores hiperparámetros RF:", gs_rf.best_params_)
print("F1 CV: {:.4f} ± {:.4f}".format(gs_rf.best_score_, gs_rf.cv_results_["std_test_score"][gs_rf.best_index_]))

r = pd.DataFrame(gs_rf.cv_results_)
r = r[r["param_clf__n_estimators"] == gs_rf.best_params_["clf__n_estimators"]]
plt.figure(figsize=(8, 5))
for d, g in r.groupby(r["param_clf__max_depth"].astype(str)):
    plt.errorbar(g["param_select__k"].astype(int), g["mean_test_score"], yerr=g["std_test_score"], marker="o", capsize=3, label=f"max_depth={d}")
plt.xlabel("k (número de características seleccionadas)"); plt.ylabel("F1 (5x2 CV)")
plt.title("Random Forest: efecto de k y max_depth"); plt.legend(); plt.tight_layout(); plt.show()

**Observación:** el mejor Random Forest usa `max_depth = 20`, 200 árboles y `k = 40`, con F1 = 0,9951 ± 0,0005, casi tres puntos por encima del MLP y con un quinto de su variabilidad. Con `max_depth = 10` los árboles son demasiado poco profundos (subajuste), sobre todo con pocas variables. El F1 **crece con k y el óptimo cae en el borde de la grilla (k = 40)**, lo que sugiere que podría mejorar con más variables. La sección 5.5 lo verifica.

### 5.3 Curvas de aprendizaje

Muestran cómo evoluciona el desempeño en entrenamiento y validación al aumentar el tamaño del conjunto de entrenamiento [5]. Permiten diagnosticar sobreajuste (brecha grande) o subajuste (ambas curvas bajas).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (name, est) in zip(axes, [("MLP", gs_mlp.best_estimator_), ("Random Forest", gs_rf.best_estimator_)]):
    sizes, tr, va = learning_curve(est, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                   scoring="f1", train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=N_JOBS)
    for s, lab, col in [(tr, "Entrenamiento", "#4C72B0"), (va, "Validación", "#C44E52")]:
        ax.plot(sizes, s.mean(1), "o-", color=col, label=lab)
        ax.fill_between(sizes, s.mean(1) - s.std(1), s.mean(1) + s.std(1), color=col, alpha=.15)
    ax.set_title(f"Curva de aprendizaje – {name}"); ax.set_xlabel("Número de muestras de entrenamiento"); ax.legend()
axes[0].set_ylabel("F1")
plt.tight_layout(); plt.show()

**Observación:**
- **MLP:** las curvas de entrenamiento y validación convergen hacia un F1 ≈ 0,97. La brecha baja de ≈ 0,015 con 2 500 muestras a ≈ 0,003 con 24 000. Las dos curvas terminan juntas y por debajo del RF, un patrón de **sesgo** (capacidad u optimización limitada) más que de varianza: agregar datos ya aporta poco y el margen de mejora está en la arquitectura y el entrenamiento.
- **Random Forest:** el F1 de entrenamiento es 1,0 (los árboles memorizan) y el de validación se estabiliza en ≈ 0,995 a partir de unas 15 000 muestras. La brecha es pequeña (≈ 0,005), así que el sobreajuste es leve y no afecta la generalización. El tamaño de muestra usado es suficiente.

### 5.4 *(Opcional – extra)* Optimización con Optuna

Búsqueda bayesiana con el muestreador TPE [6] sobre un espacio **continuo** de hiperparámetros del MLP: número de capas, neuronas por capa, `k`, `alpha` y tasa de aprendizaje. A diferencia de la búsqueda en grilla, que evalúa combinaciones fijas, TPE usa los resultados anteriores para proponer las siguientes. Para acotar el costo se usa `StratifiedKFold` de 3 folds y 20 *trials* (≈ 10–20 min en CPU). Para omitir esta sección, pon `RUN_OPTUNA = False`.

In [ ]:
RUN_OPTUNA = True
N_TRIALS = 20
if RUN_OPTUNA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=False)
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    cv_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    def objective(trial):
        n_layers = trial.suggest_int("n_layers", 1, 3)
        units = tuple(trial.suggest_categorical(f"units_{i}", [32, 64, 128, 256]) for i in range(n_layers))
        p = pipe_mlp.set_params(select__k=trial.suggest_int("k", 10, 60, step=5),
                                clf__hidden_layer_sizes=units,
                                clf__alpha=trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
                                clf__learning_rate_init=trial.suggest_float("lr", 1e-4, 1e-2, log=True))
        return cross_val_score(clone(p), X_train, y_train, cv=cv_opt, scoring="f1", n_jobs=N_JOBS).mean()
    t0 = time.time()
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"Tiempo: {time.time()-t0:.0f}s")
    print("Mejores parámetros Optuna:", study.best_params)
    print(f"F1 Optuna (3-fold CV): {study.best_value:.4f}  |  F1 GridSearch MLP (5x2 CV): "
          + (f"{gs_mlp.best_score_:.4f}" if "gs_mlp" in globals() else "n/d"))
    hist = study.trials_dataframe()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hist["number"], hist["value"], "o", alpha=.6, label="F1 de cada trial")
    ax.plot(hist["number"], hist["value"].cummax(), "-", color="#C44E52", label="Mejor F1 acumulado")
    ax.set_xlabel("Trial"); ax.set_ylabel("F1 (3-fold CV)"); ax.set_title("Historia de optimización con Optuna (TPE)")
    ax.legend(); plt.tight_layout(); plt.show()

**Observación (completar tras ejecutar):** Optuna encontró [capas/neuronas, k, alpha, lr] con F1 = [___] en 3-fold CV, frente a 0,9673 de la búsqueda en grilla. [Si es mayor:] la búsqueda en un espacio continuo, con redes más anchas, mejora al MLP, aunque sigue por debajo del Random Forest (0,9951). [Si es similar:] el resultado confirma que el techo del MLP está alrededor de 0,97 con esta familia de modelos.

### 5.5 Análisis complementarios: borde de la grilla y atajo del puerto de destino

Este análisis responde dos preguntas que surgieron en el EDA y en la optimización:
1. **¿Mejoran los modelos con más de 40 variables?** El óptimo cayó en el borde de la grilla (k = 40). Se evalúa k = 40, 55 y todas (69).
2. **¿Cuánto depende el desempeño de `Destination Port`?** Se reentrenan los mejores modelos **sin** esa variable. Si el F1 cae mucho, el modelo estaba usando el atajo del puerto [14].

Se usan los mejores hiperparámetros encontrados y `RepeatedStratifiedKFold` 5×2. La celda también funciona si solo se ejecutaron las secciones 1, 2 y 4, porque toma los valores por defecto obtenidos en la ejecución completa.

In [ ]:
mlp_best = gs_mlp.best_params_ if "gs_mlp" in globals() else {"clf__alpha": 1e-3, "clf__hidden_layer_sizes": (128, 64, 32), "select__k": 40}
rf_best  = gs_rf.best_params_  if "gs_rf"  in globals() else {"clf__max_depth": 20, "clf__n_estimators": 200, "select__k": 40}
cv_q = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
n_all = len(kept) if "kept" in globals() else X_train.shape[1]
X_noport = X_train.drop(columns=["Destination Port"])

rows = []
t0 = time.time()
for name, base, bp in [("MLP", pipe_mlp, mlp_best), ("Random Forest", pipe_rf, rf_best)]:
    for k in [40, 55, "all"]:
        est = clone(base).set_params(**{**bp, "select__k": k})
        s = cross_val_score(est, X_train, y_train, cv=cv_q, scoring="f1", n_jobs=N_JOBS)
        rows.append({"Modelo": name, "Experimento": f"k = {k if k != 'all' else f'todas ({n_all})'}", "F1 media": s.mean(), "F1 std": s.std()})
    est = clone(base).set_params(**bp)
    s = cross_val_score(est, X_noport, y_train, cv=cv_q, scoring="f1", n_jobs=N_JOBS)
    rows.append({"Modelo": name, "Experimento": f"k = {bp['select__k']}, sin Destination Port", "F1 media": s.mean(), "F1 std": s.std()})
print(f"Tiempo: {time.time()-t0:.0f}s")
extra = pd.DataFrame(rows)
display(extra.round(4))

for name in ["MLP", "Random Forest"]:
    e = extra[extra["Modelo"] == name].reset_index(drop=True)
    base_f1 = e.loc[0, "F1 media"]; best = e.iloc[:3]["F1 media"].idxmax()
    print(f"\n{name}: mejor k -> {e.loc[best, 'Experimento']} (F1 {e.loc[best, 'F1 media']:.4f}; "
          f"{(e.loc[best, 'F1 media'] - base_f1) * 100:+.2f} puntos vs k = 40)")
    print(f"{name}: sin Destination Port -> F1 {e.loc[3, 'F1 media']:.4f} ({(e.loc[3, 'F1 media'] - base_f1) * 100:+.2f} puntos)")

**Cómo leer este resultado:**
- Si ampliar `k` mejora el F1 en menos de ≈ 0,1 puntos (dentro de la desviación estándar), k = 40 es un **buen compromiso**: con 40 de 69 variables se obtiene prácticamente el mismo desempeño con menor costo.
- Si quitar `Destination Port` reduce el F1 solo marginalmente, los modelos **no dependen del atajo del puerto**: aprenden del comportamiento estadístico del flujo (tamaños, tiempos, flags), lo que da más confianza en que generalicen. Una caída grande indicaría lo contrario y obligaría a excluir el puerto en la tesis.

## 6. Comparación estadística: MLP vs. Random Forest

Se evalúan los **mejores pipelines** de cada técnica con la misma partición `RepeatedStratifiedKFold` (**10 folds × 10 repeticiones** = 100 pares de puntajes) [4]. Al compartir los folds, las observaciones son **pareadas**.

- **H₀:** no hay diferencia en el F1 de ambos modelos (la mediana de las diferencias es 0).
- **H₁:** existe diferencia.
- Prueba principal: **Wilcoxon de rangos con signo** [7], [8], no paramétrica (no asume normalidad, adecuada porque los F1 están acotados cerca de 1).
- Prueba complementaria: **t-test pareado corregido** de Nadeau y Bengio [9], que corrige la subestimación de la varianza causada por el solapamiento de los conjuntos de entrenamiento en CV repetida.
- Nivel de significancia α = 0.05.

In [ ]:
cv_cmp = RepeatedStratifiedKFold(n_splits=CMP_SPLITS, n_repeats=CMP_REPEATS, random_state=SEED + 1)
scoring = {"f1": "f1", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}
t0 = time.time()
cv_res = {name: cross_validate(est, X_train, y_train, cv=cv_cmp, scoring=scoring, n_jobs=N_JOBS)
          for name, est in [("MLP", gs_mlp.best_estimator_), ("Random Forest", gs_rf.best_estimator_)]}
print(f"Tiempo: {time.time()-t0:.0f}s")

summary = pd.DataFrame({name: {m: f"{r['test_'+m].mean():.4f} ± {r['test_'+m].std():.4f}" for m in scoring}
                        for name, r in cv_res.items()}).T
summary

In [ ]:
f1_mlp, f1_rf = cv_res["MLP"]["test_f1"], cv_res["Random Forest"]["test_f1"]
diff = f1_rf - f1_mlp

# Wilcoxon de rangos con signo (pareado)
w = stats.wilcoxon(f1_rf, f1_mlp, zero_method="wilcox", alternative="two-sided")

# t-test pareado clásico y corregido (Nadeau & Bengio, 2003)
n = len(diff); n_test = len(X_train) / CMP_SPLITS; n_tr = len(X_train) - n_test
t_naive = stats.ttest_rel(f1_rf, f1_mlp)
t_corr = diff.mean() / np.sqrt((1 / n + n_test / n_tr) * diff.var(ddof=1))
p_corr = 2 * stats.t.sf(abs(t_corr), df=n - 1)
shapiro_p = stats.shapiro(diff).pvalue

print(f"F1 medio  MLP = {f1_mlp.mean():.4f} | RF = {f1_rf.mean():.4f} | diferencia media (RF-MLP) = {diff.mean():+.4f}")
print(f"Shapiro-Wilk sobre las diferencias: p = {shapiro_p:.3g}  ({'no normal' if shapiro_p < .05 else 'normalidad no rechazada'})")
print(f"Wilcoxon:              W = {w.statistic:.1f}, p = {w.pvalue:.3g}")
print(f"t-test pareado:        t = {t_naive.statistic:.3f}, p = {t_naive.pvalue:.3g}")
print(f"t-test corregido N&B:  t = {t_corr:.3f}, p = {p_corr:.3g}")
alpha = 0.05
ganador = "Random Forest" if diff.mean() > 0 else "MLP"
print("\nConclusión (Wilcoxon, α=0.05):",
      f"se rechaza H0 -> {ganador} es significativamente mejor" if w.pvalue < alpha else "no se rechaza H0 -> no hay diferencia significativa")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
pd.DataFrame({"MLP": f1_mlp, "Random Forest": f1_rf}).plot.box(ax=axes[0], patch_artist=True)
axes[0].set_title(f"F1 en {CMP_SPLITS}x{CMP_REPEATS} CV"); axes[0].set_ylabel("F1")
sns.histplot(diff, kde=True, ax=axes[1], color="#8172B3")
axes[1].axvline(0, color="k", ls="--"); axes[1].set_title("Diferencias pareadas F1 (RF − MLP)")
plt.tight_layout(); plt.show()

**Resultado de la prueba de hipótesis:**
- En los 100 folds pareados, el F1 medio fue 0,9954 ± 0,0011 para el RF y 0,9684 ± 0,0047 para el MLP. La diferencia media es de **+0,0271** a favor del RF.
- El estadístico de Wilcoxon es **W = 0**: el Random Forest superó al MLP en **los 100 folds**, sin excepción, y el histograma de diferencias está completamente a la derecha de 0. La prueba da **p = 3,9·10⁻¹⁸ < 0,05**, así que **se rechaza H₀**.
- Shapiro-Wilk no rechaza la normalidad de las diferencias (p = 0,107), lo que también valida el t-test pareado. Incluso con la corrección de Nadeau-Bengio, que infla la varianza para compensar el solapamiento entre folds, la diferencia sigue siendo significativa (t = 17,2; p = 1,7·10⁻³¹).
- **Conclusión estadística:** con α = 0,05, el Random Forest es significativamente mejor que el MLP en este problema. Además, el MLP es menos estable (su desviación estándar es 4 veces mayor).

### 6.1 Evaluación final en el conjunto de prueba

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, est) in zip(axes, [("MLP", gs_mlp.best_estimator_), ("Random Forest", gs_rf.best_estimator_)]):
    y_pred = est.predict(X_test)
    print(f"===== {name} =====  ROC-AUC = {roc_auc_score(y_test, est.predict_proba(X_test)[:, 1]):.4f}")
    print(classification_report(y_test, y_pred, target_names=["Benigno", "Ataque"], digits=4))
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["Benigno", "Ataque"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# Recall (tasa de detección) por tipo de ataque en el conjunto de prueba
det = pd.DataFrame({"Label": lab_test, "y": y_test,
                    "MLP": gs_mlp.best_estimator_.predict(X_test),
                    "Random Forest": gs_rf.best_estimator_.predict(X_test)})
per_type = det.groupby("Label").apply(lambda g: pd.Series({
    "n": len(g),
    "MLP": (g["MLP"] == g["y"]).mean(),
    "Random Forest": (g["Random Forest"] == g["y"]).mean()}))
per_type["n"] = per_type["n"].astype(int)
per_type.sort_values("n").round(3)

La tabla muestra la proporción de flujos correctamente clasificados por tipo. Para BENIGN equivale a la especificidad; para los ataques, a la tasa de detección.

**Observación:** los resultados en el conjunto de prueba (7 626 flujos nunca vistos) confirman los de la validación cruzada. El MLP obtiene F1 = 0,9681 y ROC-AUC = 0,9950; el RF, F1 = 0,9952 y ROC-AUC = 0,9993.

| | Falsos positivos (FP) | Falsos negativos (FN) |
|---|---|---|
| **MLP** | 162 (4,1 % del tráfico benigno) | 75 (2,0 % de los ataques) |
| **RF** | 13 (0,33 % del tráfico benigno) | 22 (0,6 % de los ataques) |

En operación, la tasa de falsos positivos es crítica: con millones de flujos diarios, un 4 % de falsas alarmas haría inmanejable el monitoreo (*alert fatigue*). Por tipo de tráfico:
- Ambos modelos detectan al 100 % PortScan, DoS slowloris y DoS Slowhttptest.
- Las debilidades del MLP se concentran en **Bot** (0,913), **DoS Hulk** (0,951) y la especificidad en **BENIGN** (0,959). Bot imita tráfico legítimo, por eso es más difícil.
- Las clases con muy pocas muestras en prueba (Heartbleed n = 2, Sql Injection n = 4, Infiltration n = 7) dan estimaciones poco confiables. Infiltration es la peor para ambos modelos (0,71 y 0,57), coherente con que es un ataque sigiloso y casi sin ejemplos de entrenamiento.

## 7. Visualización con t-SNE

Se proyecta el espacio de características a 2D con t-SNE [10] sobre una submuestra de 3 000 flujos del conjunto de prueba, usando la etiqueta como color, como pide la rúbrica. Se muestran dos espacios:

1. **Características seleccionadas** (salida del filtro `SelectKBest` del pipeline).
2. **Representación aprendida** por la última capa oculta del MLP, para ver cómo la red reorganiza los datos.

In [ ]:
best_mlp = gs_mlp.best_estimator_
idx = np.random.RandomState(SEED).choice(len(X_test), size=min(N_TSNE, len(X_test)), replace=False)
Xs = X_test.iloc[idx]; ys = y_test[idx]; ls = lab_test[idx]

Z_feat = best_mlp[:-1].transform(Xs)          # salida del preprocesamiento + filtro

def hidden_repr(mlp, Z):                      # activaciones de la última capa oculta (ReLU)
    H = Z
    for W, b in zip(mlp.coefs_[:-1], mlp.intercepts_[:-1]):
        H = np.maximum(0, H @ W + b)
    return H
Z_hid = hidden_repr(best_mlp.named_steps["clf"], Z_feat)

emb = {name: TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(Z)
       for name, Z in [("Características seleccionadas", Z_feat), ("Última capa oculta del MLP", Z_hid)]}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, E) in zip(axes, emb.items()):
    for v, lab, col in [(0, "Benigno", "#4C72B0"), (1, "Ataque", "#C44E52")]:
        ax.scatter(E[ys == v, 0], E[ys == v, 1], s=6, alpha=.6, c=col, label=lab)
    ax.set_title(f"t-SNE – {name}"); ax.legend(markerscale=3); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
cats = pd.Series(ls).value_counts().index
pal = dict(zip(cats, sns.color_palette("tab20", len(cats))))
for ax, (name, E) in zip(axes, emb.items()):
    for c in cats:
        m = ls == c
        ax.scatter(E[m, 0], E[m, 1], s=6 if c == "BENIGN" else 10, alpha=.5 if c == "BENIGN" else .8, color=pal[c], label=c)
    ax.set_title(f"t-SNE por tipo de tráfico – {name}"); ax.set_xticks([]); ax.set_yticks([])
axes[1].legend(markerscale=3, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

**Interpretación:**
- **Espacio de características seleccionadas (izquierda).** El tráfico benigno forma una nube amplia y dispersa, porque agrupa muchos servicios distintos (web, DNS, correo...). Cada tipo de ataque forma, en cambio, **agrupaciones compactas y casi puras** (FTP-Patator, SSH-Patator, DoS Hulk, PortScan, DDoS): una herramienta automatizada genera flujos muy parecidos entre sí. Esto explica que ambos modelos alcancen F1 > 0,96.
- **Zonas de solapamiento.** Hay puntos de Bot y de algunos DoS mezclados con el tráfico benigno, que coinciden con las clases donde el MLP falla más. Web Attack Brute Force y Web Attack XSS comparten una misma curva alargada: con estadísticas de flujo son casi indistinguibles entre sí, lo que importará en la clasificación multiclase de la tesis.
- **Última capa oculta del MLP (derecha).** La red reorganiza el espacio y deja el tráfico benigno en la mitad izquierda y los ataques en la derecha. La separación entre clases es más clara que en el espacio original, evidencia de que la red aprende una representación orientada a la tarea. Aun así, algunos puntos benignos quedan cerca de los grupos de Bot y Web Attack, lo que corresponde a sus falsos positivos.

## 8. Conclusiones

In [ ]:
# Resumen numérico para redactar las conclusiones
print(f"Mejor MLP: {gs_mlp.best_params_}")
print(f"Mejor RF : {gs_rf.best_params_}")
print(f"F1 CV {CMP_SPLITS}x{CMP_REPEATS}: MLP {f1_mlp.mean():.4f}±{f1_mlp.std():.4f} | RF {f1_rf.mean():.4f}±{f1_rf.std():.4f}")
print(f"Wilcoxon p = {w.pvalue:.3g} | t corregido p = {p_corr:.3g}")
for name, est in [("MLP", gs_mlp.best_estimator_), ("RF", gs_rf.best_estimator_)]:
    print(f"F1 test {name}: {f1_score(y_test, est.predict(X_test)):.4f}")

1. **El problema es altamente separable con estadísticas de flujo.** Ambos modelos superan F1 = 0,96 en validación cruzada y en prueba. El t-SNE muestra por qué: cada herramienta de ataque produce flujos homogéneos que forman grupos compactos. Las dificultades se concentran en los ataques que imitan tráfico legítimo (Bot, Infiltration) y en las clases con muy pocas muestras.
2. **La limpieza del dataset es indispensable.** CIC-IDS2017 trae valores infinitos y NaN, 8 columnas constantes, una columna duplicada, 2 064 flujos duplicados (casi todos ataques) y 31 pares de variables con |ρ| > 0,95. Integrar el tratamiento en un `Pipeline` evitó la fuga de información en los cientos de entrenamientos de la validación cruzada.
3. **La selección de características reduce el modelo sin perder desempeño.** Con el filtro ANOVA-F, ambos modelos alcanzan su mejor resultado con 40 de las 69 variables útiles (58 %). Bajar a 10–20 variables sí degrada el F1, sobre todo en el MLP (≈ 1,3 puntos menos con k = 20). [Completar con 5.5: ampliar k a 55 o 69 mejora el F1 en ___ puntos.]
4. **El Random Forest es significativamente mejor que la red neuronal en este escenario.** En validación cruzada 10×10, el RF obtuvo F1 = 0,9954 ± 0,0011 y el MLP 0,9684 ± 0,0047. El RF ganó en los 100 folds (Wilcoxon W = 0, p = 3,9·10⁻¹⁸; t corregido de Nadeau-Bengio p = 1,7·10⁻³¹). En el conjunto de prueba, el MLP generó 12 veces más falsas alarmas (162 frente a 13), una diferencia decisiva para un IDS en producción.
5. **El MLP está limitado por sesgo, no por falta de datos.** Su curva de aprendizaje converge en F1 ≈ 0,97 con una brecha mínima, la regularización casi no influye y la arquitectura más profunda siempre fue mejor. Mejorarlo exige más capacidad y mejor entrenamiento (más épocas, *batch normalization*, *dropout*, ajuste de la tasa de aprendizaje), no más datos.
6. **El desempeño no depende del atajo del puerto.** [Completar con 5.5: sin `Destination Port` el F1 cambia en ___ puntos (MLP) y ___ (RF).] Si el cambio es pequeño, los modelos aprenden del comportamiento del tráfico y no de la configuración del experimento.
7. **Trabajo futuro para la tesis:**
   - Arquitecturas profundas más adecuadas (MLP con *dropout* y *batch normalization* en PyTorch Lightning, 1D-CNN), con el objetivo de cerrar la brecha con el RF.
   - Clasificación multiclase por tipo de ataque, con manejo explícito de las clases raras (*class weights*, SMOTE).
   - Validación cruzada entre datasets (por ejemplo, entrenar en CIC-IDS2017 y evaluar en UNSW-NB15) para medir la generalización real.

## 9. Referencias

[1] I. Sharafaldin, A. H. Lashkari y A. A. Ghorbani, "Toward generating a new intrusion detection dataset and intrusion traffic characterization," en *Proc. 4th Int. Conf. Information Systems Security and Privacy (ICISSP)*, Funchal, Portugal, 2018, pp. 108–116.

[2] G. Engelen, V. Rimmer y W. Joosen, "Troubleshooting an intrusion detection dataset: the CICIDS2017 case study," en *Proc. IEEE Security and Privacy Workshops (SPW)*, 2021, pp. 7–12.

[3] F. Pedregosa *et al.*, "Scikit-learn: Machine learning in Python," *J. Mach. Learn. Res.*, vol. 12, pp. 2825–2830, 2011.

[4] scikit-learn developers, "RepeatedStratifiedKFold," *scikit-learn 1.9 documentation*. [En línea]. Disponible: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedStratifiedKFold.html

[5] scikit-learn developers, "learning_curve," *scikit-learn 1.9 documentation*. [En línea]. Disponible: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html

[6] T. Akiba, S. Sano, T. Yanase, T. Ohta y M. Koyama, "Optuna: A next-generation hyperparameter optimization framework," en *Proc. 25th ACM SIGKDD Int. Conf. Knowledge Discovery & Data Mining*, 2019, pp. 2623–2631.

[7] F. Wilcoxon, "Individual comparisons by ranking methods," *Biometrics Bulletin*, vol. 1, no. 6, pp. 80–83, 1945.

[8] J. Demšar, "Statistical comparisons of classifiers over multiple data sets," *J. Mach. Learn. Res.*, vol. 7, pp. 1–30, 2006.

[9] C. Nadeau y Y. Bengio, "Inference for the generalization error," *Machine Learning*, vol. 52, no. 3, pp. 239–281, 2003.

[10] L. van der Maaten y G. Hinton, "Visualizing data using t-SNE," *J. Mach. Learn. Res.*, vol. 9, pp. 2579–2605, 2008.

[11] scikit-learn developers, "Feature selection," *scikit-learn 1.9 documentation*. [En línea]. Disponible: https://scikit-learn.org/stable/modules/feature_selection.html

[12] The SciPy community, "scipy.stats.wilcoxon," *SciPy documentation*. [En línea]. Disponible: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html

[13] Canadian Institute for Cybersecurity, "Intrusion Detection Evaluation Dataset (CIC-IDS2017)," Univ. of New Brunswick. [En línea]. Disponible: https://www.unb.ca/cic/datasets/ids-2017.html

[14] D. Arp *et al.*, "Dos and don'ts of machine learning in computer security," en *Proc. 31st USENIX Security Symposium*, 2022, pp. 3971–3988.